# 01 — Data: Validation Harness, then Real Data

**Phase 1.** Build the pipeline, prove it recovers known ground truth on synthetic
data, then point it at real surfaces — which is where the project actually lives.

The roles are inverted on purpose:

- **Synthetic data is the validation harness.** Simulate the stochastic heat
  equation, run the *entire* downstream pipeline on it, and confirm we recover what
  we put in (sinusoidal modes, $\lambda_k \sim k^{-2}$, the correct $D$). This proves
  the *code* is correct — it is **not** a finding.
- **Real options data is the study.** The research question is where real surfaces
  match the diffusive-membrane picture and where they deviate.

## 1.1 — Synthetic data as the validation harness (not the study)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from src.simulate import StochasticHeatEquation, SPDEConfig
from src.plotting import apply_style, COLORS

apply_style()
print('Setup complete.')

In [ ]:
# Simulate ~8 years of daily data on a moneyness grid z in [-3, 3].
config = SPDEConfig(D=0.05, n_z=50, z_min=-3.0, z_max=3.0, n_modes=25, seed=42)
spde = StochasticHeatEquation(config)

surface = spde.simulate(n_days=2000)
returns = spde.compute_returns(surface)
print(f'surface: {surface.shape}   returns: {returns.shape}')

Watch the harness output as a vibrating string over time.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(surface[:500].T, aspect='auto', cmap='RdBu_r',
               extent=[0, 500, config.z_min, config.z_max], origin='lower')
plt.colorbar(im, ax=ax, label='perturbation p(z,t)')
ax.set_xlabel('trading day'); ax.set_ylabel('moneyness z')
ax.set_title('Synthetic surface perturbation (first 500 days)')
plt.tight_layout(); plt.show()

A quick ground-truth check before any PCA: correlation should decay with distance in z.

In [ ]:
corr = np.corrcoef(returns.T)
mid = len(spde.z_grid) // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0]); axes[0].set_title('return correlation matrix')
axes[1].plot(np.abs(spde.z_grid - spde.z_grid[mid]), corr[mid], 'o-', ms=3)
axes[1].set_xlabel('|z - z_mid|'); axes[1].set_ylabel('correlation')
axes[1].set_title('correlation vs distance')
plt.tight_layout(); plt.show()

Save the harness dataset for the downstream notebooks.

In [ ]:
import os
os.makedirs('../data/synthetic', exist_ok=True)
np.savez('../data/synthetic/spde_dataset.npz',
         z_grid=spde.z_grid, surface=surface, returns=returns,
         D=config.D, n_modes=config.n_modes, noise_amplitude=config.noise_amplitude)
print('saved ../data/synthetic/spde_dataset.npz')

## 1.2 — Acquire real options data (the actual deliverable)

Deep historical full-surface options data is expensive, but viable routes exist
(see `src/data_pipeline.py` and the README):

1. **WRDS / OptionMetrics (IvyDB)** — cleanest path to real equity surfaces; free if
   your institution subscribes. **Check this first.**
2. **Deribit BTC/ETH options** — free public API, several years of liquid history.
3. **yfinance** — current chains only; a sanity check, not a basis for the study.

Pick **one** source you can actually get and commit to it in Week 1 — every later
phase depends on it.

In [ ]:
# TODO (Phase 1): load a real source, then run the identical pipeline.
# The loaders below are scaffolding in src/data_pipeline.py:
#
#   from src.data_pipeline import load_optionmetrics, load_deribit, preprocess_options_data
#   raw = load_deribit(currency='BTC', start='2021-01-01')      # -> standard schema
#   df  = preprocess_options_data(raw)                          # adds (z, tau, T_years)
#
# The synthetic harness above and the real data below must flow through the SAME code.
pass

## Next steps

- **02_reparam** — the $(z,\tau)$ reparameterization, with the $\sqrt{T}$ fix.
- **03_pca** — identical PCA on synthetic *and* real, side by side.
- **04_physics** — eigenvalue scaling; where real data leaves $k^{-2}$ is the finding.